In [ ]:
import kairo
import pandas as pd

import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv
from pathlib import Path
import os

load_dotenv(Path("../..") / ".env")

In [ ]:
log = kairo.read_log("../../data/drifted_logs/time_fast/sudden/bank_account_closure/bank_account_closure_time_fast_x05_sudden.xes",
    case_id = "case:concept:name",
    activity = "concept:name",
    timestamp = "time:timestamp",
    #event_id = "event:uidd",
    start_timestamp = "start:timestamp",
    resource = "org:resource",
    #event_duration = "event:runtime_min")
    )
log.head()

In [ ]:
stats = kairo.compute_log_stats(log)
stats

In [ ]:
activity_plot = kairo.plot_activity_counts(stats)
activity_plot.show()

In [ ]:
# one row per calendar window, the case attributes with their kind,
# e.g. {"case:amount": "numerical", "case:region": "categorical"}
features = kairo.compute_features_inter(log, window="1D",
    case_attributes={"CLOSURE_TYPE": "categorical"},
    stall_threshold="1D")
features

In [ ]:
std_features = kairo.standardize(features, method="minmax")
std_features

In [ ]:
pca = kairo.compute_pca(std_features, num_components=None)
pca_plot = kairo.plot_pca_variances(pca)
pca_plot.show()
#pca

In [ ]:
reconstruction_errors = kairo.compute_pca_reconstruction_errors(std_features, pca)
reconstruction_plot = kairo.plot_pca_reconstruction_errors(reconstruction_errors)
reconstruction_plot.show()

In [ ]:
compressed_features = kairo.apply_pca(std_features, pca, cut_component=5)
compressed_features

In [ ]:
pca_plot_cut = kairo.plot_pca_variances(pca, cut_component=5)
pca_plot_cut.show()

In [ ]:
reconstruction_plot_cut = kairo.plot_pca_reconstruction_errors(reconstruction_errors, cut_component=5)
reconstruction_plot_cut.show()

In [ ]:
kmeans = kairo.compute_kmeans(compressed_features, k=3)

In [ ]:
clusters = kairo.get_kmeans_clusters(compressed_features, kmeans)
clusters

In [ ]:
clustering_scores = [
    {"type": "Silhouette", "value": kairo.compute_silhouette(compressed_features, clusters)},
    {"type": "Calinski-Harabasz", "value": kairo.compute_calinski_harabasz(compressed_features, clusters)},
]
clustering_scores

In [ ]:
features_and_clusters = pd.concat([compressed_features, clusters], axis=1)
features_and_clusters

In [ ]:
state_distances = kairo.get_kmeans_state_distances(kmeans)
state_distances

In [ ]:
state_freqs = kairo.get_kmeans_state_frequencies(features_and_clusters)
state_freqs

In [ ]:
distances_plot = kairo.plot_kmeans_distances(state_distances)
distances_plot.show()

In [ ]:
color_mapping = kairo.compute_kmeans_color_mapping(3)

In [ ]:
freqs_plot = kairo.plot_kmeans_frequencies(state_freqs, color_mapping)
freqs_plot.show()

In [ ]:
log_trajectory = kairo.get_kmeans_log_trajectory(features_and_clusters,
                                 #start_date="2017-06-01",
                                 #end_date="2018-11-26"
                                 )
log_trajectory

In [ ]:
log_trajectory_plot = kairo.plot_kmeans_log_trajectory(log_trajectory, color_mapping)
log_trajectory_plot

In [ ]:
# the states of the daily windows counted per 30 days
freq_distributions = kairo.compute_state_distributions(features_and_clusters, window="30D")
freq_distributions

In [ ]:
distributions_plot = kairo.plot_state_distributions(freq_distributions, color_mapping)
distributions_plot.show()

In [ ]:
divergences = kairo.compute_divergences(freq_distributions)
divergences

In [ ]:
divergences_plot = kairo.plot_divergences(divergences)
divergences_plot.show()

In [ ]:
window_distances = kairo.compute_window_distances(compressed_features, distance="euclidean", reference="previous")
window_distances

In [ ]:
window_distances_plot = kairo.plot_window_distances(window_distances, distance="euclidean", reference="previous")
window_distances_plot.show()

In [ ]:
abs_log_stats = kairo.abstract_log_stats(stats)
abs_log_stats

In [ ]:
abs_features = kairo.abstract_features(features)
abs_features

In [ ]:
abs_pca = kairo.abstract_pca(pca, cut_component=5)
abs_pca

In [ ]:
abs_states = kairo.abstract_states(state_freqs, state_distances, color_mapping, clustering_scores=clustering_scores)
abs_states

In [ ]:
abs_log_trajectory = kairo.abstract_log_trajectory(log_trajectory)
abs_log_trajectory

In [ ]:
abs_distributions = kairo.abstract_distributions(freq_distributions)
abs_distributions

In [ ]:
abs_divergences = kairo.abstract_divergences(divergences, divergence="kl", reference="previous")
abs_divergences

In [ ]:
abs_window_distances = kairo.abstract_window_distances(window_distances, distance="euclidean", reference="previous")
abs_window_distances

In [ ]:
llm = kairo.LLMConnector(
    provider="custom",
    #api_key=os.environ["ANTHROPIC_API_KEY"],
    model="Qwen/Qwen3-VL-32B-Instruct",
    base_url="http://137.226.117.70:8000/v1"
)

In [ ]:
chat_history = []
total_out_tokens = 0

In [ ]:
# For initial request
messages = kairo.create_messages(provider="custom",
                                system_prompt=kairo.DEFAULT_SYSTEM_PROMPT,
                                prompts=[abs_log_stats, abs_features, abs_pca, abs_states, abs_log_trajectory,
                                         abs_distributions, abs_divergences, abs_window_distances,
                                         "Analyze and summarize the given information. Do not use markdown format, just raw text"],
                                plots=[activity_plot, pca_plot_cut, distances_plot, freqs_plot,
                                       log_trajectory_plot, distributions_plot, divergences_plot, window_distances_plot])
chat_history += messages

num_tokens = kairo.count_input_tokens(provider="custom", messages=chat_history)
num_tokens

In [ ]:
# For follow up request
messages = kairo.create_messages(provider="custom",
                                system_prompt=None,
                                prompts=["what was my previous question very briefly"],
                                plots=[])
chat_history += messages

num_tokens = kairo.count_input_tokens(provider="custom", messages=chat_history)
num_tokens

In [ ]:
response = llm.call(messages=chat_history, max_tokens=64000)

answer = kairo.get_response_text(response)

chat_history.append({"role": "assistant", 
                     "content": answer})

print(answer)

In [ ]:
out_tokens = kairo.count_output_tokens("custom", response)
total_out_tokens += out_tokens
total_out_tokens

In [ ]:
print(kairo.get_chat_history(chat_history))